# Loading the data:

In [1]:
# load the cleaned dataset
#from ReusableFunc import load_clean_data


#PrData = load_clean_data(r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\ProData.csv")

#print("Data loaded successfully!")
import pandas as pd
clean_data=pd.read_csv(r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\ProcessedData.csv")
print("Data Loaded Successfully")

Data Loaded Successfully


In [2]:
clean_data.isnull().sum()

Patient_ID             0
Age                    0
Gender                 0
Region                 0
Insurance_Type         0
Admission_Type         0
Hospital_Department    0
Length_of_Stay         0
Previous_Admissions    0
Previous_ER_Visits     0
Diabetes               0
Hypertension           0
Heart_Disease          0
Medication_Count       0
Lab_Test_Count         0
Average_Glucose        0
Systolic_BP            0
Discharge_Type         0
Followup_Scheduled     0
Followup_Attended      0
Treatment_Cost         0
Satisfaction_Score     0
Readmitted_30_Days     0
dtype: int64

# Statistical Analysis

# 1. What is the typical patients Age, Length of stay and Treatment cost?

In [9]:
clean_data[['Age', 'Length_of_Stay', 'Treatment_Cost']].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,500.0,53.990,20.743831,18.0,37.0,54.0,72.00,90.0
Length_of_Stay,500.0,4.678,4.258472,1.0,2.0,4.0,6.00,60.0
Treatment_Cost,500.0,85775.874,32339.752884,17712.0,62532.0,81396.5,104412.25,193348.0


Descriptive analysis was conducted to understand the distribution of key numerical variables. The average patient age was 53.99 years (SD = 20.74), with ages ranging from 18 to 90 years. 

Hospital length of stay averaged 4.68 days (SD = 4.25), with a median of 4 days and a range of 1 to 60 days. The maximum length of stay of 60 days appears substantially higher than the typical stay and should be investigated as a potential outlier. 

Treatment costs averaged approximately KSh 85,775 and a standard deviation of KSh 32,339, indicating considerable variation in treatment expenditure.

# 2. Is Readmission associated with Diabetes, gender, Admission type and discharge type?

In [10]:
import pandas as pd

# Create a crosstab to analyze the relationship between Diabetes and Readmission within 30 days
pd.crosstab(
    clean_data['Diabetes'],
    clean_data['Readmitted_30_Days'],
    normalize='index'
).mul(100).round(2)

# Create a crosstab to analyze the relationship between Gender and Readmission within 30 days
#pd.crosstab(
   # PrData['Gender'],
    #PrData['Readmitted_30_Days'],
   # normalize='index'
#).mul(100).round(2) 

Readmitted_30_Days,No,Yes
Diabetes,,
No,79.51,20.49
Yes,66.42,33.58


Diabetes vs 30-Day Readmission

The cross-tabulation indicates a difference in 30-day readmission between patients with and without diabetes. Among patients without diabetes, 20.49% were readmitted within 30 days, while 79.51% were not readmitted. In contrast, among patients with diabetes, 33.58% were readmitted and 66.42% were not readmitted. This suggests that patients with diabetes had a higher proportion of 30-day readmissions compared with patients without diabetes.

Key finding
Diabetes	Not readmitted	Readmitted
No	79.51%	20.49%
Yes	66.42%	33.58%

So, readmission was about 13.09 percentage points higher among patients with diabetes (33.58% vs 20.49%).

However, this is only a descriptive finding. To determine whether the relationship between diabetes and readmission is statistically significant, you should follow it with a chi-square test of independence.

# 3. Chi-square test

The chi-square test is appropriate when you want to determine whether two categorical variables are statistically associated.

For example:

Diabetes vs 30-day readmission

In [11]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    clean_data['Diabetes'],
    clean_data['Readmitted_30_Days']
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

Chi-square: 8.511176680589005
p-value: 0.003529717454583244


A chi-square test of independence was conducted to assess the association between diabetes status and 30-day hospital readmission. The test showed a statistically significant association between diabetes and 30-day readmission (χ² = 8.511, p = 0.003). Patients with diabetes had a higher readmission rate (33.58%) compared with patients without diabetes (20.49%). Therefore, diabetes status appears to be associated with 30-day readmission in this dataset.

In conclusion: Diabetes was significantly associated with 30-day readmission

# 5. Correlation Analysis:

Does length of stay increase as treatment cost increase?

In [12]:
clean_data[['Length_of_Stay', 'Treatment_Cost']].corr()


,Length_of_Stay,Treatment_Cost
Length_of_Stay,1.000000,0.640875
Treatment_Cost,0.640875,1.000000


There was a moderately strong positive correlation between Length of Stay and Treatment Cost (r = 0.641). This indicates that patients with longer hospital stays tended to incur higher treatment costs. The finding suggests that length of stay may be an important factor associated with treatment expenditure. However, correlation does not imply causation, and other patient and treatment-related factors may also influence treatment costs.

# group comparison:


In [13]:
# Summary statistics by readmission status

summary = clean_data.groupby('Readmitted_30_Days')[
    ['Length_of_Stay', 'Treatment_Cost']
].mean()

print(summary.round(2))

                    Length_of_Stay  Treatment_Cost
Readmitted_30_Days                                
No                            4.52        84564.54
Yes                           5.17        89611.76


Interpretation

Patients who were readmitted within 30 days had an average hospital stay of 5.18 days, compared with 4.52 days among patients who were not readmitted. This represents a difference of approximately 0.66 days.

Similarly, patients who were readmitted had a higher average treatment cost of approximately KSh 89,611, compared with KSh 84,564 among patients who were not readmitted. This represents a difference of approximately KSh 5,047.

Overall finding

The descriptive analysis suggests that patients who experienced 30-day readmission tended to have longer hospital stays and higher treatment costs than those who were not readmitted. This may indicate that length of stay and treatment cost are associated with readmission outcomes. However, these differences alone do not establish statistical significance or causation.

To confirm this we need to test whether these differences are statistically significant using a t-test (or Mann–Whitney U test) for Length of Stay and Treatment Cost between the two readmission groups.

# Independent Samples t-test:
Does Length of Stay and Treatment Cost differ significantly between patients who were readmitted and those who were not?

use an independent samples t-test. If the data are not normally distributed, use the Mann–Whitney U test.

# Normality Test:

In [14]:
from scipy.stats import shapiro

# Length of Stay
los_no = clean_data.loc[clean_data['Readmitted_30_Days'] == 'No', 'Length_of_Stay'].dropna()
los_yes = clean_data.loc[clean_data['Readmitted_30_Days'] == 'Yes', 'Length_of_Stay'].dropna()

print("Length of Stay - Not Readmitted")
print(shapiro(los_no))

print("\nLength of Stay - Readmitted")
print(shapiro(los_yes))


# Treatment Cost
cost_no = clean_data.loc[clean_data['Readmitted_30_Days'] == 'No', 'Treatment_Cost'].dropna()
cost_yes = clean_data.loc[clean_data['Readmitted_30_Days'] == 'Yes', 'Treatment_Cost'].dropna()

print("\nTreatment Cost - Not Readmitted")
print(shapiro(cost_no))

print("\nTreatment Cost - Readmitted")
print(shapiro(cost_yes))

Length of Stay - Not Readmitted
ShapiroResult(statistic=np.float64(0.5941608598097735), pvalue=np.float64(7.336756265094407e-29))

Length of Stay - Readmitted
ShapiroResult(statistic=np.float64(0.636302757351092), pvalue=np.float64(8.242366149913809e-16))

Treatment Cost - Not Readmitted
ShapiroResult(statistic=np.float64(0.9724184082454735), pvalue=np.float64(1.2798337811022448e-06))

Treatment Cost - Readmitted
ShapiroResult(statistic=np.float64(0.948926395331642), pvalue=np.float64(0.00018038109788834868))


Not normally distributed , use mann-whitney-u

In [15]:
from scipy.stats import mannwhitneyu

# Length of Stay
u_los, p_los_mw = mannwhitneyu(
    los_no,
    los_yes,
    alternative='two-sided'
)

print("Length of Stay - Mann-Whitney U")
print("U-statistic:", u_los)
print("p-value:", p_los_mw)


# Treatment Cost
u_cost, p_cost_mw = mannwhitneyu(
    cost_no,
    cost_yes,
    alternative='two-sided'
)

print("\nTreatment Cost - Mann-Whitney U")
print("U-statistic:", u_cost)
print("p-value:", p_cost_mw)

Length of Stay - Mann-Whitney U
U-statistic: 21371.5
p-value: 0.2965259870353891

Treatment Cost - Mann-Whitney U
U-statistic: 21658.0
p-value: 0.40806463635332424


A Mann–Whitney U test was conducted to assess whether Length of Stay and Treatment Cost differed significantly between patients who were readmitted within 30 days and those who were not. For Length of Stay, the test produced a U-statistic of 21,371 and a p-value of 0.296. Since the p-value was greater than 0.05, there was no statistically significant difference in Length of Stay between the two groups.

For Treatment Cost, the Mann–Whitney U test produced a U-statistic of 21,658 and a p-value of 0.408. Since the p-value was also greater than 0.05, there was no statistically significant difference in Treatment Cost between readmitted and non-readmitted patients.

Although readmitted patients had a higher average Length of Stay (5.18 days versus 4.52 days) and higher average Treatment Cost (KSh 89,891.17 versus KSh 84,601.39), these observed differences were not statistically significant at the 5% significance level. 

# Conclusion:
Therefore, the analysis does not provide sufficient statistical evidence that Length of Stay or Treatment Cost is associated with 30-day readmission in this dataset.


# Statistical Analysis test to determine variables associated with 30 days readmission.


Categorical variables vs 30 days readmission:
Run Chi-Square test of association

In [3]:
categorical_vars = [
    'Gender',
    'Region',
    'Insurance_Type',
    'Admission_Type',
    'Hospital_Department',
    'Diabetes',
    'Hypertension',
    'Heart_Disease',
    'Discharge_Type',
    'Followup_Scheduled',
    'Followup_Attended'
]

In [4]:
# Automating the test
from scipy.stats import chi2_contingency

def chi_square_test(clean_data, variable, target='Readmitted_30_Days'):
    
    table = pd.crosstab(clean_data[variable], clean_data[target])
    
    chi2, p_value, dof, expected = chi2_contingency(table)
    
    print(f"\n{variable} - Chi-square Test")
    print("-" * 45)
    print("Chi-square statistic:", chi2)
    print("p-value:", p_value)
    
    if p_value < 0.05:
        print("Conclusion: Statistically significant association")
    else:
        print("Conclusion: No statistically significant association")
    
    return chi2, p_value

In [5]:
# Running the test
chi_results = {}

for variable in categorical_vars:
    chi2, p = chi_square_test(clean_data, variable)
    chi_results[variable] = {
        'chi2': chi2,
        'p_value': p
    }


Gender - Chi-square Test
---------------------------------------------
Chi-square statistic: 10.752007504846832
p-value: 0.0046262727697206135
Conclusion: Statistically significant association

Region - Chi-square Test
---------------------------------------------
Chi-square statistic: 6.578658275570994
p-value: 0.36157398635922205
Conclusion: No statistically significant association

Insurance_Type - Chi-square Test
---------------------------------------------
Chi-square statistic: 0.4122551606053317
p-value: 0.9814612959503701
Conclusion: No statistically significant association

Admission_Type - Chi-square Test
---------------------------------------------
Chi-square statistic: 4.4148915712995285
p-value: 0.10998120563516063
Conclusion: No statistically significant association

Hospital_Department - Chi-square Test
---------------------------------------------
Chi-square statistic: 5.049615852544244
p-value: 0.40985508432583406
Conclusion: No statistically significant association


# Saving into a table


In [6]:
import pandas as pd
from scipy.stats import chi2_contingency

categorical_vars = [
    'Gender',
    'Region',
    'Insurance_Type',
    'Admission_Type',
    'Hospital_Department',
    'Diabetes',
    'Hypertension',
    'Heart_Disease',
    'Discharge_Type',
    'Followup_Scheduled',
    'Followup_Attended'
]

categorical_results = []

for variable in categorical_vars:

    # Contingency table
    table = pd.crosstab(
        clean_data[variable],
        clean_data['Readmitted_30_Days']
    )

    # Chi-square test
    chi2, p_value, dof, expected = chi2_contingency(table)

    # Add result
    categorical_results.append({
        'Variable': variable,
        'Chi-square': chi2,
        'p-value': p_value,
        'Significant': 'Yes' if p_value < 0.05 else 'No'
    })

categorical_results_df = pd.DataFrame(categorical_results)

categorical_results_df = categorical_results_df.sort_values(
    'p-value'
)

categorical_results_df.round(4)

,Variable,Chi-square,p-value,Significant
5,Diabetes,8.5112,0.0035,Yes
0,Gender,10.7520,0.0046,Yes
8,Discharge_Type,6.6825,0.0827,No
3,Admission_Type,4.4149,0.1100,No
6,Hypertension,1.0086,0.3152,No
10,Followup_Attended,2.2390,0.3264,No
1,Region,6.5787,0.3616,No
9,Followup_Scheduled,0.7164,0.3973,No
4,Hospital_Department,5.0496,0.4099,No
7,Heart_Disease,0.0908,0.7632,No


# Numerical Variables vs 30 days Readmission

In [7]:
numerical_vars = [
    'Age',
    'Length_of_Stay',
    'Previous_Admissions',
    'Previous_ER_Visits',
    'Medication_Count',
    'Lab_Test_Count',
    'Average_Glucose',
    'Systolic_BP',
    'Treatment_Cost',
    'Satisfaction_Score'
]

# Note
Don't necessarily need to run Shapiro-Wilk on every variable to decide whether to use Mann-Whitney. With hospital data, variables such as treatment cost and length of stay are often skewed, and your sample is large enough that normality tests can become overly sensitive.

use Mann-Whitney U for these numerical variables.

In [8]:
from scipy.stats import mannwhitneyu

def mann_whitney_test(clean_data, variable, target='Readmitted_30_Days'):
    
    group_no = clean_data.loc[
        clean_data[target] == 'No', variable
    ].dropna()
    
    group_yes = clean_data.loc[
        clean_data[target] == 'Yes', variable
    ].dropna()
    
    u_stat, p_value = mannwhitneyu(
        group_no,
        group_yes,
        alternative='two-sided'
    )
    
    print(f"\n{variable} - Mann-Whitney U Test")
    print("-" * 45)
    print("U-statistic:", u_stat)
    print("p-value:", p_value)
    
    if p_value < 0.05:
        print("Conclusion: Statistically significant difference")
    else:
        print("Conclusion: No statistically significant difference")
    
    return u_stat, p_value

In [9]:
# Running the test
mw_results = {}

for variable in numerical_vars:
    u, p = mann_whitney_test(clean_data, variable)
    
    mw_results[variable] = {
        'U_statistic': u,
        'p_value': p
    }


Age - Mann-Whitney U Test
---------------------------------------------
U-statistic: 18867.5
p-value: 0.004369109131128006
Conclusion: Statistically significant difference

Length_of_Stay - Mann-Whitney U Test
---------------------------------------------
U-statistic: 21371.5
p-value: 0.2965259870353891
Conclusion: No statistically significant difference

Previous_Admissions - Mann-Whitney U Test
---------------------------------------------
U-statistic: 18519.5
p-value: 0.0010905540547059328
Conclusion: Statistically significant difference

Previous_ER_Visits - Mann-Whitney U Test
---------------------------------------------
U-statistic: 21993.5
p-value: 0.5443932071595683
Conclusion: No statistically significant difference

Medication_Count - Mann-Whitney U Test
---------------------------------------------
U-statistic: 22724.0
p-value: 0.9560077947676076
Conclusion: No statistically significant difference

Lab_Test_Count - Mann-Whitney U Test
--------------------------------------

# Saving in tables

In [10]:
#import pandas as pd
#from scipy.stats import mannwhitneyu

numerical_results = []

for variable in numerical_vars:

    # Separate readmission groups
    group_no = clean_data.loc[
        clean_data['Readmitted_30_Days'] == 'No',
        variable
    ].dropna()

    group_yes = clean_data.loc[
        clean_data['Readmitted_30_Days'] == 'Yes',
        variable
    ].dropna()

    # Mann-Whitney U test
    u_stat, p_value = mannwhitneyu(
        group_no,
        group_yes,
        alternative='two-sided'
    )

    numerical_results.append({
        'Variable': variable,
        'U-statistic': u_stat,
        'p-value': p_value,
        'Significant': 'Yes' if p_value < 0.05 else 'No'
    })

# Create DataFrame
numerical_results_df = pd.DataFrame(numerical_results)

# Sort by p-value
numerical_results_df = numerical_results_df.sort_values('p-value')

# Display table
numerical_results_df.round(4)

,Variable,U-statistic,p-value,Significant
2,Previous_Admissions,18519.5,0.0011,Yes
0,Age,18867.5,0.0044,Yes
6,Average_Glucose,21270.0,0.2676,No
9,Satisfaction_Score,24277.5,0.2838,No
1,Length_of_Stay,21371.5,0.2965,No
5,Lab_Test_Count,24168.0,0.3196,No
8,Treatment_Cost,21658.0,0.4081,No
7,Systolic_BP,21778.0,0.4590,No
3,Previous_ER_Visits,21993.5,0.5444,No
4,Medication_Count,22724.0,0.9560,No


# Readmission proportion calculation for every categorical variables.

In [11]:
# Function to calculate
def calculate_readmission_proportions(clean_data, variable, target='Readmitted_30_Days'):
    
    table = pd.crosstab(
        clean_data[variable],
        clean_data[target],
        normalize='index'
    ) * 100
    
    print(f"\n{variable} - Readmission Proportions (%)")
    print("-" * 50)
    print(table.round(2))
    
    return table

In [12]:
# Running it for all categorical variables
categorical_vars = [
    'Gender',
    'Region',
    'Insurance_Type',
    'Admission_Type',
    'Hospital_Department',
    'Diabetes',
    'Hypertension',
    'Heart_Disease',
    'Discharge_Type',
    'Followup_Scheduled',
    'Followup_Attended'
]

for variable in categorical_vars:
    calculate_readmission_proportions(
        clean_data,
        variable
    )


Gender - Readmission Proportions (%)
--------------------------------------------------
Readmitted_30_Days     No    Yes
Gender                          
Male                81.47  18.53
female              72.44  27.56
other               50.00  50.00

Region - Readmission Proportions (%)
--------------------------------------------------
Readmitted_30_Days     No    Yes
Region                          
Central             72.37  27.63
Coast               75.31  24.69
Eastern             77.78  22.22
Nairobi             69.12  30.88
Nyanza              86.89  13.11
Rift Valley         77.46  22.54
Western             74.65  25.35

Insurance_Type - Readmission Proportions (%)
--------------------------------------------------
Readmitted_30_Days     No    Yes
Insurance_Type                  
Cash                75.00  25.00
Employer            78.95  21.05
NHIF                76.00  24.00
Private             75.49  24.51
Unknown             71.43  28.57

Admission_Type - Readmission Pr

# Saving proportions in a table

In [13]:
proportion_results = []

for variable in categorical_vars:
    
    table = pd.crosstab(
        clean_data[variable],
        clean_data['Readmitted_30_Days'],
        normalize='index'
    ) * 100
    
    for category in table.index:
        
        proportion_results.append({
            'Variable': variable,
            'Category': category,
            'Not_Readmitted_%': table.loc[category].get('No', 0),
            'Readmitted_%': table.loc[category].get('Yes', 0)
        })

proportion_df = pd.DataFrame(proportion_results)

proportion_df.round(2)

,Variable,Category,Not_Readmitted_%,Readmitted_%
0,Gender,Male,81.47,18.53
1,Gender,female,72.44,27.56
2,Gender,other,50.00,50.00
3,Region,Central,72.37,27.63
4,Region,Coast,75.31,24.69
5,Region,Eastern,77.78,22.22
6,Region,Nairobi,69.12,30.88
7,Region,Nyanza,86.89,13.11
8,Region,Rift Valley,77.46,22.54
9,Region,Western,74.65,25.35


# Numerical Variables:
compute median for the addmitted and non-readdmitted groups

In [14]:
# function to calculate medians
def calculate_medians(clean_data, variable, target='Readmitted_30_Days'):
    
    summary = clean_data.groupby(target)[variable].agg(
        Median='median',
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    
    summary['IQR'] = summary['Q3'] - summary['Q1']
    
    print(f"\n{variable} - Median and IQR")
    print("-" * 50)
    print(summary.round(2))
    
    return summary

In [15]:
# Running it for all numerical variables
numerical_vars = [
    'Age',
    'Length_of_Stay',
    'Previous_Admissions',
    'Previous_ER_Visits',
    'Medication_Count',
    'Lab_Test_Count',
    'Average_Glucose',
    'Systolic_BP',
    'Treatment_Cost',
    'Satisfaction_Score'
]

median_results = {}

for variable in numerical_vars:
    median_results[variable] = calculate_medians(
        clean_data,
        variable
    )


Age - Median and IQR
--------------------------------------------------
                    Median     Q1    Q3    IQR
Readmitted_30_Days                            
No                    52.0  35.00  70.0  35.00
Yes                   59.0  39.75  76.0  36.25

Length_of_Stay - Median and IQR
--------------------------------------------------
                    Median   Q1   Q3  IQR
Readmitted_30_Days                       
No                     4.0  2.0  6.0  4.0
Yes                    4.0  2.0  7.0  5.0

Previous_Admissions - Median and IQR
--------------------------------------------------
                    Median   Q1   Q3  IQR
Readmitted_30_Days                       
No                     1.0  0.0  2.0  2.0
Yes                    1.0  1.0  2.0  1.0

Previous_ER_Visits - Median and IQR
--------------------------------------------------
                    Median   Q1   Q3  IQR
Readmitted_30_Days                       
No                     1.0  1.0  2.0  1.0
Yes             

# Saving the report in the table


In [16]:
median_results = []

for variable in numerical_vars:
    
    grouped = clean_data.groupby('Readmitted_30_Days')[variable]
    
    for group_name, values in grouped:
        
        median_results.append({
            'Variable': variable,
            'Group': group_name,
            'Median': values.median(),
            'Q1': values.quantile(0.25),
            'Q3': values.quantile(0.75),
            'IQR': values.quantile(0.75) - values.quantile(0.25)
        })

median_df = pd.DataFrame(median_results)

median_df.round(2)

,Variable,Group,Median,Q1,Q3,IQR
0,Age,No,52.00,35.00,70.00,35.00
1,Age,Yes,59.00,39.75,76.00,36.25
2,Length_of_Stay,No,4.00,2.00,6.00,4.00
3,Length_of_Stay,Yes,4.00,2.00,7.00,5.00
4,Previous_Admissions,No,1.00,0.00,2.00,2.00
5,Previous_Admissions,Yes,1.00,1.00,2.00,1.00
6,Previous_ER_Visits,No,1.00,1.00,2.00,1.00
7,Previous_ER_Visits,Yes,1.00,1.00,2.00,1.00
8,Medication_Count,No,6.00,4.00,7.00,3.00
9,Medication_Count,Yes,6.00,4.00,7.00,3.00
